# Stage 2 — Cell Segmentation with CellViT

**Pipeline context:**
```
Stage 1 (local): patchify.py → processed/he/*.png  ←── you upload these
Stage 2 (this notebook): CellViT inference → processed/cellvit/*.json
Stage 3 (local): assign_cells.py → cell type + state masks
```

**What this notebook does:**
1. Installs CellViT and dependencies
2. Downloads the CellViT-256 checkpoint (~400 MB)
3. Reads your `processed/he/` patches from Google Drive or AWS S3
4. Runs CellViT inference (GPU) on each 256×256 H&E patch
5. Post-processes outputs → cell centroids, contours, and type predictions
6. Saves `processed/cellvit/{i}_{j}.json` back to Google Drive or S3

**Before running:**
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Upload your `processed/he/` folder to Google Drive under `he-feature-visualizer/processed/he/`,
  OR upload to S3 and configure the S3 settings below.

## 1 — Configuration
Edit the paths in this cell before running anything else.

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────

# Storage backend: 'drive' (Google Drive) or 's3' (AWS S3)
STORAGE_BACKEND = 'drive'   # change to 's3' to use S3

# --- Google Drive paths (used when STORAGE_BACKEND == 'drive') ---
DRIVE_HE_DIR  = 'he-feature-visualizer/processed/he'
DRIVE_OUT_DIR = 'he-feature-visualizer/processed/cellvit'

# --- AWS S3 settings (used when STORAGE_BACKEND == 's3') ---
S3_BUCKET     = 'your-bucket-name'
S3_HE_PREFIX  = 'he-feature-visualizer/processed/he'
S3_OUT_PREFIX = 'he-feature-visualizer/processed/cellvit'
S3_REGION     = 'us-east-1'

# Model variant
# 'CellViT-256'   — 256×256 patches at 40x (default; matches Stage 1 patch size)
# 'CellViT-SAM-H' — higher quality, ~6× slower, needs more VRAM
MODEL_VARIANT = 'CellViT-256'

# Inference settings
BATCH_SIZE    = 32    # patches per forward pass; reduce to 8 if you get OOM
MAGNIFICATION = 40    # 40 for CellViT-256; use 20 for CellViT-256-x20
NUM_WORKERS   = 2

# Skip patches that already have a JSON output (useful to resume interrupted runs)
SKIP_EXISTING = True

# ── DERIVED (do not edit) ─────────────────────────────────────────────────────
import os
COLAB_HE_DIR = '/content/he_patches'     # local copy for fast I/O during inference
CELLVIT_DIR  = '/content/CellViT'
CKPT_DIR     = '/content/checkpoints'

CHECKPOINT_GDRIVE_IDS = {
    'CellViT-256':   '1tVYAapUo1Xt8QgCN22Ne1urbbCZkah8q',
    'CellViT-SAM-H': '1MvRKNzDW2eHbQb5rAgTEp6s2zAXHixRV',
}
CHECKPOINT_FILE = os.path.join(CKPT_DIR, f'{MODEL_VARIANT}.pth')

print(f'Model:            {MODEL_VARIANT}')
print(f'Storage backend:  {STORAGE_BACKEND}')
print(f'Batch size:       {BATCH_SIZE}')

## 2 — Storage Setup (Google Drive or AWS S3)

In [ ]:
import os

if STORAGE_BACKEND == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive'
    HE_DIR  = os.path.join(DRIVE_ROOT, DRIVE_HE_DIR)
    OUT_DIR = os.path.join(DRIVE_ROOT, DRIVE_OUT_DIR)
    assert os.path.isdir(HE_DIR), f'Not found: {HE_DIR}\nUpload patches to Drive: MyDrive/{DRIVE_HE_DIR}'
    patches_available = sorted(p for p in os.listdir(HE_DIR) if p.endswith('.png'))
    print(f'Found {len(patches_available)} patches in Drive.')

elif STORAGE_BACKEND == 's3':
    # Install and configure boto3 + AWS CLI
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'boto3'], check=True)
    import boto3

    # Credentials — enter here OR set as Colab secrets (Secrets panel on left sidebar)
    # Recommended: use Colab Secrets to avoid exposing keys in notebook
    try:
        from google.colab import userdata
        AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
        AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
    except Exception:
        # Fallback: prompt user
        import getpass
        AWS_ACCESS_KEY_ID     = getpass.getpass('AWS_ACCESS_KEY_ID: ')
        AWS_SECRET_ACCESS_KEY = getpass.getpass('AWS_SECRET_ACCESS_KEY: ')

    os.environ['AWS_ACCESS_KEY_ID']     = AWS_ACCESS_KEY_ID
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
    os.environ['AWS_DEFAULT_REGION']    = S3_REGION

    s3_client = boto3.client('s3', region_name=S3_REGION)

    # List available patches from S3
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_HE_PREFIX + '/')
    patches_available = sorted(
        os.path.basename(obj['Key'])
        for page in pages
        for obj in page.get('Contents', [])
        if obj['Key'].endswith('.png')
    )
    print(f'Found {len(patches_available)} patches in s3://{S3_BUCKET}/{S3_HE_PREFIX}')
    HE_DIR  = COLAB_HE_DIR  # local SSD; S3 patches are always copied locally
    OUT_DIR = '/content/cellvit_output'
    os.makedirs(OUT_DIR, exist_ok=True)

print('First few patches:', patches_available[:5])

## 3 — Install CellViT
This clones the repo and installs its dependencies. Takes ~2 minutes.

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Runtime settings!')

# Clone CellViT
if not os.path.isdir(CELLVIT_DIR):
    !git clone --quiet https://github.com/TIO-IKIM/CellViT.git {CELLVIT_DIR}
    print('Cloned CellViT.')
else:
    print('CellViT already cloned.')

# Add to Python path
if CELLVIT_DIR not in sys.path:
    sys.path.insert(0, CELLVIT_DIR)

# Install requirements (suppress most output)
!pip install -q -r {CELLVIT_DIR}/requirements.txt

# Patch pydantic version conflict (CellViT requires pydantic 1.x, Colab may have 2.x)
!pip install -q 'pydantic==1.10.4'

print('Installation done.')

## 4 — Download Model Checkpoint
CellViT-256 is ~400 MB; CellViT-SAM-H is ~2.5 GB.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.isfile(CHECKPOINT_FILE):
    size_mb = os.path.getsize(CHECKPOINT_FILE) / 1e6
    print(f'Checkpoint already exists ({size_mb:.0f} MB): {CHECKPOINT_FILE}')
else:
    file_id = CHECKPOINT_GDRIVE_IDS[MODEL_VARIANT]
    print(f'Downloading {MODEL_VARIANT} checkpoint …')
    !pip install -q gdown
    import gdown
    gdown.download(id=file_id, output=CHECKPOINT_FILE, quiet=False)
    size_mb = os.path.getsize(CHECKPOINT_FILE) / 1e6
    print(f'Downloaded: {size_mb:.0f} MB')

# Quick sanity check
import torch
ckpt_meta = torch.load(CHECKPOINT_FILE, map_location='cpu')
print('Checkpoint keys:', list(ckpt_meta.keys())[:8])
run_conf = ckpt_meta.get('run_conf', {})
print('Backbone:', run_conf.get('model', {}).get('backbone', 'not found in run_conf'))

## 5 — Download Patches to Local SSD
Drive/S3 I/O is slow; copying patches locally speeds up inference ~10×.

In [ ]:
import shutil
from tqdm.auto import tqdm

os.makedirs(COLAB_HE_DIR, exist_ok=True)

if STORAGE_BACKEND == 'drive':
    to_copy = [p for p in patches_available
               if not os.path.isfile(os.path.join(COLAB_HE_DIR, p))]
    if to_copy:
        print(f'Copying {len(to_copy)} patches from Drive ...')
        for fname in tqdm(to_copy):
            shutil.copy2(os.path.join(HE_DIR, fname),
                         os.path.join(COLAB_HE_DIR, fname))

elif STORAGE_BACKEND == 's3':
    to_download = [p for p in patches_available
                   if not os.path.isfile(os.path.join(COLAB_HE_DIR, p))]
    if to_download:
        print(f'Downloading {len(to_download)} patches from S3 ...')
        for fname in tqdm(to_download):
            s3_key = f'{S3_HE_PREFIX}/{fname}'
            s3_client.download_file(S3_BUCKET, s3_key,
                                    os.path.join(COLAB_HE_DIR, fname))

local_patches = sorted(p for p in os.listdir(COLAB_HE_DIR) if p.endswith('.png'))
print(f'Local patches ready: {len(local_patches)}')

## 6 — Load Model

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Inference will be extremely slow.')

# ── Load checkpoint & rebuild model ──────────────────────────────────────────────
checkpoint = torch.load(CHECKPOINT_FILE, map_location=DEVICE)
run_conf   = checkpoint['run_conf']
backbone   = run_conf['model'].get('backbone', 'CellViT256')
print(f'Backbone from checkpoint: {backbone}')

# Select model class based on backbone name in checkpoint
if 'SAM' in backbone.upper():
    from cell_segmentation.models.cellvit.cellvit_sam import CellViTSAM
    model_cls = CellViTSAM
    model_kwargs = dict(
        model_path         = run_conf['model'].get('pretrained_encoder', ''),
        num_nuclei_classes = run_conf['data']['num_nuclei_classes'],
        num_tissue_classes = run_conf['data']['num_tissue_classes'],
        vit_structure      = run_conf['model'].get('backbone', 'SAM-H').split('-', 1)[-1].lower(),
        drop_rate          = run_conf['training'].get('drop_rate', 0),
    )
else:  # CellViT256 (default)
    from cell_segmentation.models.cellvit.cellvit import CellViT256
    model_cls = CellViT256
    model_kwargs = dict(
        model_path         = run_conf['model'].get('pretrained_encoder', ''),
        num_nuclei_classes = run_conf['data']['num_nuclei_classes'],
        num_tissue_classes = run_conf['data']['num_tissue_classes'],
        drop_rate          = run_conf['training'].get('drop_rate', 0),
        attn_drop_rate     = run_conf['training'].get('attn_drop_rate', 0),
        drop_path_rate     = run_conf['training'].get('drop_path_rate', 0),
    )

model = model_cls(**model_kwargs)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

NUM_NUCLEI_CLASSES = run_conf['data']['num_nuclei_classes']
print(f'Model loaded. Nuclei classes: {NUM_NUCLEI_CLASSES}')

# ── Cell type mapping (PanNuke) ──────────────────────────────────────────────
CELL_TYPE_NAMES = {
    0: 'background',
    1: 'Neoplastic',
    2: 'Inflammatory',
    3: 'Connective',
    4: 'Dead',
    5: 'Epithelial',
}

# ── Image transforms (ImageNet normalisation, matching CellViT training) ──────
inference_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

## 7 — Run Inference
Processes all patches in batches. Each batch takes ~0.5 s on a T4 GPU.  
For 1000 patches this takes roughly 15–20 minutes.

In [ ]:
import json
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from cell_segmentation.utils.post_proc_cellvit import DetectionCellPostProcessor

os.makedirs(OUT_DIR, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────────────────────────────
class PatchDataset(Dataset):
    """Loads 256×256 H&E PNG patches."""
    def __init__(self, patch_dir, fnames, transform):
        self.patch_dir = patch_dir
        self.fnames    = fnames
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = Image.open(os.path.join(self.patch_dir, fname)).convert('RGB')
        return self.transform(img), fname

# Decide which patches to process
if SKIP_EXISTING:
    done = {os.path.splitext(f)[0] for f in os.listdir(OUT_DIR) if f.endswith('.json')}
    todo = [f for f in local_patches
            if os.path.splitext(f)[0] not in done]
    print(f'Skipping {len(done)} already-processed patches.')
else:
    todo = local_patches

print(f'Patches to process: {len(todo)}')

dataset = PatchDataset(COLAB_HE_DIR, todo, inference_transform)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=(DEVICE.type == 'cuda'), shuffle=False)

# ── Post-processor ──────────────────────────────────────────────────────────────────
post_proc = DetectionCellPostProcessor(
    nr_types      = NUM_NUCLEI_CLASSES,
    magnification = MAGNIFICATION,
    gt            = False,
)

# ── Inference loop ──────────────────────────────────────────────────────────────────
total_cells = 0

with torch.inference_mode():
    for imgs, fnames in tqdm(loader, desc='Patches'):
        imgs = imgs.to(DEVICE)        # (B, 3, 256, 256)

        # ── Forward pass ──────────────────────────────────────────────────────
        outputs = model(imgs)
        # outputs keys: 'nuclei_binary_map' (B,2,H,W), 'hv_map' (B,2,H,W),
        #               'nuclei_type_map'   (B,C,H,W), 'tissue_types' (B,T)

        # ── Assemble pred_map (B, H, W, 4) for post-processor ─────────────────
        # Ch 0 : nuclei probability  (sigmoid on binary logits, foreground channel)
        # Ch 1 : HV map X
        # Ch 2 : HV map Y
        # Ch 3 : cell type (argmax)
        nuc_prob  = torch.sigmoid(outputs['nuclei_binary_map'])[:, 1, :, :]  # (B,H,W)
        hv_x      = outputs['hv_map'][:, 0, :, :]                            # (B,H,W)
        hv_y      = outputs['hv_map'][:, 1, :, :]                            # (B,H,W)
        type_pred = torch.argmax(
            torch.softmax(outputs['nuclei_type_map'], dim=1), dim=1          # (B,H,W)
        ).float()

        pred_maps = torch.stack([nuc_prob, hv_x, hv_y, type_pred], dim=-1)  # (B,H,W,4)
        pred_maps = pred_maps.cpu().numpy().astype(np.float32)

        # ── Per-patch post-processing and JSON save ────────────────────────────
        for b, fname in enumerate(fnames):
            patch_id = os.path.splitext(fname)[0]   # e.g. '3_7'

            _, cell_info = post_proc.post_process_cell_segmentation(pred_maps[b])
            # cell_info: dict of {cell_id: {centroid, contour, bbox, type, type_prob}}

            cells = []
            for cdata in cell_info.values():
                cell_type_id = int(cdata['type'])
                cells.append({
                    'centroid':     [round(float(v), 2) for v in cdata['centroid']],
                    'contour':      [[int(x), int(y)] for x, y in cdata['contour']],
                    'bbox':         [[int(v) for v in row] for row in cdata['bbox']],
                    'type_cellvit': cell_type_id,
                    'type_name':    CELL_TYPE_NAMES.get(cell_type_id, 'unknown'),
                    'type_prob':    round(float(cdata.get('type_prob', 0.0)), 4),
                })

            out_path = os.path.join(OUT_DIR, f'{patch_id}.json')
            with open(out_path, 'w') as f:
                json.dump({'patch': patch_id, 'cells': cells}, f)

            total_cells += len(cells)

print(f'\nDone. Processed {len(todo)} patches, {total_cells:,} cells total.')
print(f'JSON files saved to: {OUT_DIR}')

## 8 — Verify Output
Spot-check one patch result and visualise cells on the H&E.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba

TYPE_COLORS = {
    0: 'gray',
    1: 'red',       # Neoplastic
    2: 'blue',      # Inflammatory
    3: 'green',     # Connective
    4: 'black',     # Dead
    5: 'orange',    # Epithelial
}

# Pick first available JSON
json_files = sorted(f for f in os.listdir(OUT_DIR) if f.endswith('.json'))
assert json_files, 'No JSON files found in output directory.'

sample_id = os.path.splitext(json_files[0])[0]
with open(os.path.join(OUT_DIR, json_files[0])) as f:
    result = json.load(f)

he_path = os.path.join(COLAB_HE_DIR, f'{sample_id}.png')
if not os.path.isfile(he_path):
    he_path = os.path.join(HE_DIR, f'{sample_id}.png')  # fallback to Drive

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
he_img = np.array(Image.open(he_path).convert('RGB'))

# Left: raw H&E
axes[0].imshow(he_img)
axes[0].set_title(f'H&E \u2014 patch {sample_id}')
axes[0].axis('off')

# Right: H&E + cell centroids coloured by CellViT type
axes[1].imshow(he_img)
cells = result['cells']
for cell in cells:
    cx, cy = cell['centroid']
    color = TYPE_COLORS.get(cell['type_cellvit'], 'white')
    axes[1].plot(cx, cy, '.', color=color, markersize=4, alpha=0.8)

legend_handles = [
    mpatches.Patch(color=TYPE_COLORS[t], label=f'{CELL_TYPE_NAMES[t]}')
    for t in range(1, 6)
]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=7)
axes[1].set_title(f'CellViT detections \u2014 {len(cells)} cells')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Summary stats
from collections import Counter
type_counts = Counter(c['type_name'] for c in cells)
print(f'\nPatch {sample_id}: {len(cells)} cells')
for name, count in sorted(type_counts.items()):
    print(f'  {name:<15} {count:>4}')

## 9 — Upload Results to Drive / S3
Syncs completed JSON files back to your chosen storage backend.

In [ ]:
json_files_out = [f for f in os.listdir(OUT_DIR) if f.endswith('.json')]
print(f'JSON files produced: {len(json_files_out)}')

if STORAGE_BACKEND == 's3':
    print(f'Uploading {len(json_files_out)} JSON files to s3://{S3_BUCKET}/{S3_OUT_PREFIX}/ ...')
    for fname in tqdm(json_files_out):
        s3_key = f'{S3_OUT_PREFIX}/{fname}'
        s3_client.upload_file(os.path.join(OUT_DIR, fname), S3_BUCKET, s3_key)
    print('Upload complete.')
    print(f'\nDownload locally with:')
    print(f'  aws s3 sync s3://{S3_BUCKET}/{S3_OUT_PREFIX}/ processed/cellvit/')
elif STORAGE_BACKEND == 'drive':
    print(f'Results already saved to Drive: {DRIVE_OUT_DIR}')
    print(f'\nDownload locally with rclone:')
    print(f'  rclone copy gdrive:{DRIVE_OUT_DIR}/ processed/cellvit/ --progress')

# Progress check
total_input = len(patches_available)
remaining = total_input - len(json_files_out)
if remaining > 0:
    print(f'\nRemaining: {remaining} patches (re-run inference cell to continue)')
else:
    print('All patches processed!')

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: pydantic` | Re-run cell 3 (install); restart runtime if needed |
| `CUDA out of memory` | Reduce `BATCH_SIZE` to 8 or 4 in the config cell |
| `KeyError: 'nuclei_binary_map'` | Checkpoint does not match expected output format — try `CellViT-256` instead of SAM variant |
| `model_state_dict` missing | Checkpoint may be just weights; replace `checkpoint['model_state_dict']` with `checkpoint` |
| Drive I/O timeout | Re-run cell 5 again; Colab Drive can drop connections |
| `DetectionCellPostProcessor` import fails | Re-run cell 3 and ensure `sys.path` includes `/content/CellViT` |
| `NoCredentialsError` (S3) | Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` to Colab Secrets (left sidebar) |
| `NoSuchBucket` (S3) | Check `S3_BUCKET` name and `S3_REGION` in the config cell |
| `AccessDenied` (S3) | Ensure your IAM user/role has `s3:GetObject`, `s3:PutObject`, and `s3:ListBucket` permissions |
| S3 listing returns 0 patches | Verify `S3_HE_PREFIX` matches the exact S3 key prefix where you uploaded the `.png` files |

**After downloading results:**  
Copy `processed/cellvit/` to your local `he-feature-visualizer/processed/cellvit/` and run Stage 3:
```bash
python assign_cells.py \
  --cellvit-dir processed/cellvit/ \
  --features-csv data/CRC02.csv \
  --out processed/
```